<a href="https://colab.research.google.com/github/JuanZapa7a/AINavalEngineering/blob/main/NB11_Neural_Network_Fundamentals_First_PyTorch_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **NB11 · Class 11 — Neural Network Fundamentals: Your First PyTorch Model**

## Block 3: AI — Deep Learning (opening)

`NB01`–`NB10` covered classical Machine Learning: models that work directly on hand-engineered numeric features. This class opens **Block 3 — Deep Learning**: neural networks, which learn their own internal representations instead of relying entirely on features we chose by hand. We build understanding from the smallest possible unit (a single perceptron) up to a real, trained multi-layer network — using **[PyTorch](https://pytorch.org/)**, a widely used deep learning framework we'll rely on throughout this block.

To keep the comparison honest and concrete, we train our first network on a dataset we already know well: the real **Sonar (Mines vs. Rocks)** dataset from `NB08`, and compare our neural network directly against `NB08`'s classical results (Decision Tree, Random Forest, AdaBoost, SVM).

### Learning objectives

By the end of this class, students will be able to:
- Explain what a perceptron computes, and why a single perceptron can only learn linearly separable patterns.
- Explain how stacking layers (a Multi-Layer Perceptron) and non-linear activation functions overcome that limitation.
- Explain, at a conceptual level, how a network learns: forward pass, loss function, backpropagation, gradient descent.
- Build a small feedforward neural network in PyTorch (`nn.Module`, a training loop with `loss.backward()`/`optimizer.step()`).
- Train the network, plot its loss curve, and evaluate it exactly like any other classifier from `NB08`.

### Agenda (2-hour class)

| # | Class segment | Approx. duration | Type |
|---|---------------------|:---:|:---:|
| 1 | Recap of Block 2, roadmap for Block 3 | 5 min | Theory |
| 2 | Why Deep Learning? Where classical ML hits a ceiling | 10 min | Theory |
| 3 | The perceptron: the basic unit | 15 min | Theory |
| 4 | From perceptron to Multi-Layer Perceptron: layers and activation functions | 10 min | Theory |
| 5 | How a network learns: loss, backpropagation, gradient descent | 15 min | Theory |
| 6 | PyTorch basics: tensors and loading our real dataset | 15 min | Practice |
| 7 | Hands-on: building and training a first MLP classifier | 25 min | Practice |
| 8 | Evaluating our network and comparing it to NB08's classical models | 15 min | Practice |
| 9 | A first look at overfitting in neural networks | 5 min | Theory |
| 10 | Summary, homework, next class | 5 min | Theory |

> Timings are approximate guidance, not a strict script — there are no scheduled breaks. If we cover everything with time to spare, class ends early; that can happen and is fine.


---

## 1. Recap: where we are

- **Block 1** (`NB01`): AI history and context.
- **Block 2** (`NB02`–`NB10`): the full classical ML toolkit — Python/NumPy/Pandas, the supervised workflow, trees/ensembles/SVM, unsupervised learning, and one complete tuned project.
- **Block 3** (starting today): Deep Learning — neural networks, built up from first principles and trained for real.

This is the block that fills the biggest gap identified when this course was compared against its official guía docente: Deep Learning had no hands-on content at all before this redesign.

---

## 2. Why Deep Learning? Where classical ML hits a ceiling

Every model in Block 2 worked on **tabular, hand-engineered features**: `distance`, `engine_efficiency`, 60 sonar frequency bands, hull geometry coefficients. A human (or a data pipeline) decided in advance which numbers mattered.

That works well when:
- The data is already naturally tabular (spreadsheets, sensor logs, structured records).
- A domain expert can name the relevant features (e.g., naval engineers already know Froude number matters for resistance).

It works less well when:
- The raw data is **unstructured** — an image, an audio waveform, raw text, a long sensor sequence — where "the right features" aren't obvious, or there are far too many possible ones to hand-engineer.
- The patterns that matter are **hierarchical**: an edge → a shape → an object, in an image; a phoneme → a word → a sentence, in audio/text.

**Neural networks** address this by `learning their own intermediate representations directly from raw(er) data, layer by layer`, instead of requiring us to hand-design every feature. This is *why* the next few classes exist: `NB13` (CNNs) will apply this directly to real underwater inspection images, where hand-engineering "the right features" would be far harder than it was for our tabular sonar/ship datasets.

Today, though, we deliberately start on **familiar tabular data** (Sonar, same as `NB08`) so the *only* new variable is the model itself — everything else (the data, the evaluation) stays comparable to what you already know.

---

## 3. The perceptron: the basic unit

A **[perceptron](https://en.wikipedia.org/wiki/Perceptron)** (Rosenblatt, 1958) is the simplest possible neural network: it takes several numeric inputs, multiplies each by a learned **weight**, adds a **bias**, and passes the result through an **activation function**:

$$
z = w_1 x_1 + w_2 x_2 + \dots + w_n x_n + b \qquad \hat{y} = f(z)
$$

Let's put that structure into a picture — a single perceptron with 4 example inputs, a bias term, and one output:

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

fig, ax = plt.subplots(figsize=(8, 5))

input_labels = ["x1", "x2", "x3", "x4"]
input_y = [4, 3, 2, 1]
input_x = 0
neuron_x, neuron_y = 3.5, 2.5

# Input nodes
for label, y in zip(input_labels, input_y):
    ax.add_patch(patches.Circle((input_x, y), 0.3, facecolor="lightblue", edgecolor="black", zorder=3))
    ax.text(input_x, y, label, ha="center", va="center", fontsize=11, zorder=4)

# Bias node (drawn separately, feeds in like an extra input)
bias_y = 0
ax.add_patch(patches.Circle((input_x, bias_y), 0.3, facecolor="khaki", edgecolor="black", zorder=3))
ax.text(input_x, bias_y, "b", ha="center", va="center", fontsize=11, zorder=4)

# Neuron: weighted sum + activation
ax.add_patch(patches.Circle((neuron_x, neuron_y), 0.55, facecolor="lightcoral", edgecolor="black", zorder=3))
ax.text(neuron_x, neuron_y, "Σ  f", ha="center", va="center", fontsize=12, zorder=4)

# Arrows from inputs to the neuron, labeled with weights
for label, y, w in zip(input_labels, input_y, ["w1", "w2", "w3", "w4"]):
    ax.annotate("", xy=(neuron_x - 0.55, neuron_y), xytext=(input_x + 0.3, y),
                arrowprops=dict(arrowstyle="->", lw=1.5))
    ax.text((input_x + neuron_x) / 2 - 0.3, (y + neuron_y) / 2, w, fontsize=10, color="darkblue")

# Bias arrow (dashed, to distinguish it from the real inputs)
ax.annotate("", xy=(neuron_x - 0.55, neuron_y), xytext=(input_x + 0.3, bias_y),
            arrowprops=dict(arrowstyle="->", lw=1.5, linestyle="dashed", color="darkgoldenrod"))

# Output arrow
output_x = 6.5
ax.annotate("", xy=(output_x, neuron_y), xytext=(neuron_x + 0.55, neuron_y),
            arrowprops=dict(arrowstyle="->", lw=2))
ax.text(output_x + 0.2, neuron_y, "y_hat", fontsize=13, va="center")

ax.set_xlim(-1, 8)
ax.set_ylim(-1, 5)
ax.axis("off")
ax.set_title("Structure of a single perceptron")
plt.show()

Each input $x_i$ is multiplied by its own weight $w_i$ (the solid arrows); the bias $b$ (dashed arrow) shifts the result independently of any input; the neuron adds everything up ($\Sigma$) and applies the activation function $f$ to produce the output $\hat{y}$. **The weights and bias are exactly what training learns** — the diagram's *shape* (how many inputs, how the arrows connect) is fixed by the architecture we choose; the *numbers on the arrows* are what gradient descent adjusts.

If $f$ is a step function, this is exactly **Logistic Regression** from `NB07` in disguise (if $f$ is the logistic/sigmoid function, it's *precisely* Logistic Regression) — a useful anchor: a single perceptron is not a new idea, `it's the neural-network way of expressing a model you already understand`.

**The catch**: a single perceptron can only separate data with a straight line (or hyperplane) — it can only learn **linearly separable** patterns. Many real problems (including, likely, our sonar data — recall SVM needed a non-linear RBF kernel in `NB08` to do well) are not linearly separable. That limitation is exactly what motivates the next section.

> **Further reading**: [Perceptron (Wikipedia)](https://en.wikipedia.org/wiki/Perceptron).

---

## 4. From perceptron to Multi-Layer Perceptron

Stack perceptrons into **layers**, and feed each layer's output as the next layer's input, and you get a **Multi-Layer Perceptron (MLP)**: an input layer, one or more **hidden layers**, and an output layer.

One subtlety matters enormously: if every layer only computes a weighted sum (no non-linearity), stacking layers is mathematically pointless — `any chain of linear functions collapses back into a single linear function`, no more powerful than one perceptron. **Non-linear activation functions** between layers are what actually give depth its power.

| Activation | Formula | Typical use |
|---|---|---|
| **Sigmoid** | $\dfrac{1}{1+e^{-z}}$ | Output layer for binary classification (squashes to 0–1, like `NB07`'s Logistic Regression) |
| **ReLU** (Rectified Linear Unit) | $\max(0, z)$ | The default choice for hidden layers — fast, simple, avoids a training problem sigmoid has in deep networks |
| **Tanh** | $\dfrac{e^z - e^{-z}}{e^z + e^{-z}}$ | Similar to sigmoid, but centered at 0 |

We'll use **ReLU** in our hidden layers (standard practice) and a **sigmoid-equivalent** output for our binary mine/rock prediction.

> **Further reading**: [Multilayer perceptron (Wikipedia)](https://en.wikipedia.org/wiki/Multilayer_perceptron) · [Activation function (Wikipedia)](https://en.wikipedia.org/wiki/Activation_function).

---

## 5. How a network learns

Training a network means finding weights and biases that make its predictions match reality as closely as possible. Three ingredients:

1. **A loss function** measures how wrong a prediction is. For binary classification (our sonar task), we use **binary cross-entropy** — conceptually similar to the classification metrics from `NB07`, but differentiable, which matters for step 3.
2. **Backpropagation** computes, for every weight in the network, *how much that specific weight contributed to the current error* — efficiently, by applying the chain rule of calculus backwards through the network, layer by layer.
3. **Gradient descent** then nudges every weight slightly in the direction that reduces the loss, repeats over many small steps (**epochs**), and — if all goes well — the loss falls and predictions improve.

$$
w \leftarrow w - \eta \frac{\partial \text{Loss}}{\partial w}
$$

where $\eta$ (the **learning rate**) controls how big each step is: `too large and training can diverge; too small and training crawls`. In practice, we don't compute any of this by hand — PyTorch's `loss.backward()` handles backpropagation automatically, and an **optimizer** (we'll use **Adam**, a refined version of gradient descent) handles the weight updates.

> **Further reading**: [Backpropagation (Wikipedia)](https://en.wikipedia.org/wiki/Backpropagation) · [Gradient descent (Wikipedia)](https://en.wikipedia.org/wiki/Gradient_descent) · [Cross-entropy (Wikipedia)](https://en.wikipedia.org/wiki/Cross-entropy).

---

## 6. PyTorch basics: tensors and our real dataset

PyTorch's core data structure is the **tensor** — like a NumPy array (`NB02`), but with two extra abilities we need for deep learning: it can track gradients automatically (for backpropagation), and it can run on a GPU. Colab ships with PyTorch pre-installed, so we can import it directly.

In [ ]:
import torch
import torch.nn as nn

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

Reload the real Sonar dataset — same file, same columns, same task as `NB08`:

In [ ]:
!wget -q -O sonar.csv https://raw.githubusercontent.com/JuanZapa7a/AINavalEngineering/main/Datasets/sonar.all-data

import pandas as pd

sonar = pd.read_csv("sonar.csv", header=None)
sonar.columns = [f"freq_{i}" for i in range(60)] + ["label"]

X = sonar.drop(columns="label").values
y = (sonar["label"] == "M").astype(int).values
print(X.shape, y.shape)

Split and scale exactly as in `NB08` — neural networks are, if anything, *more* sensitive to unscaled inputs than SVM was, since large input values can make training unstable:

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

Convert the NumPy arrays into PyTorch tensors — the format every PyTorch model expects:

In [ ]:
X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
X_test_t = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)

X_train_t.shape, y_train_t.shape

---

## 7. Hands-on: building and training a first MLP classifier

Define the network: 60 inputs (our frequency bands) → a hidden layer of 32 units → a hidden layer of 16 units → 1 output (mine probability). This is a small, deliberately simple architecture — a starting point, not a tuned final model.

In [ ]:
class SonarMLP(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(n_features, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
        )

    def forward(self, x):
        return self.layers(x)

model = SonarMLP(n_features=X_train_t.shape[1])
model

We output a raw score ("logit") rather than a 0–1 probability directly — `BCEWithLogitsLoss` applies the sigmoid and computes the loss together, in a single, more numerically stable step. Define the loss function and the optimizer:

In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

Now the training loop — the heart of deep learning, and worth reading line by line the first time: for each epoch, compute predictions (forward pass), compute the loss, compute gradients (backward pass), and update the weights.

In [ ]:
n_epochs = 200
train_losses = []

for epoch in range(n_epochs):
    optimizer.zero_grad()
    outputs = model(X_train_t)
    loss = criterion(outputs, y_train_t)
    loss.backward()
    optimizer.step()

    train_losses.append(loss.item())
    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch + 1}/{n_epochs} - training loss: {loss.item():.4f}")

Plot the loss curve — this should fall steadily as training progresses:

In [ ]:
import matplotlib.pyplot as plt

plt.plot(train_losses)
plt.xlabel("Epoch")
plt.ylabel("Training loss (binary cross-entropy)")
plt.title("MLP training loss over epochs")
plt.show()

**Read your own curve**: a smoothly decreasing loss that flattens out is exactly what we want — the network is learning, then converging. A loss that jumps around wildly usually means the learning rate is too high; a loss that barely moves usually means it's too low (or the network is too small for the pattern).

---

## 8. Evaluating our network and comparing it to `NB08`

Switch the model to evaluation mode (disables training-only behavior we'll meet in `NB12`) and predict on the held-out test set, exactly as we would for any `NB08` classifier:

In [ ]:
model.eval()
with torch.no_grad():
    test_logits = model(X_test_t)
    test_probs = torch.sigmoid(test_logits)
    test_preds = (test_probs > 0.5).float()

accuracy = (test_preds == y_test_t).float().mean().item()
print("Test accuracy:", round(accuracy, 3))

Get the full picture with the same tools as `NB08`:

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

y_pred_np = test_preds.numpy().ravel()
y_test_np = y_test_t.numpy().ravel()

print(confusion_matrix(y_test_np, y_pred_np))
print()
print(classification_report(y_test_np, y_pred_np, target_names=["Rock", "Mine"]))

**Compare to `NB08`**: how does this untuned MLP's accuracy compare to the Decision Tree / Random Forest / AdaBoost / SVM boxplot from `NB08`'s Part 7? It's common for a first, untuned neural network to land somewhere in the middle of the pack, not automatically ahead — `neural networks generally need more data and more careful tuning than tree-based methods to show a real advantage`, which is exactly why `NB12` exists.

---

## 9. A first look at overfitting in neural networks

Recall `NB08`'s decision-tree depth experiment: more capacity isn't free. Neural networks have the same failure mode, at a different knob: more layers, more units per layer, and more training epochs all `increase a network's capacity to memorize its training data rather than learn the underlying pattern`.

With only 208 sonar examples and a 60→32→16→1 network (thousands of trainable weights), we are already in a regime where overfitting is a real risk over long enough training — worth keeping an eye on the gap between training loss and test performance. `NB12` covers this properly: validation curves, dropout, and early stopping, the neural-network equivalents of `NB08`'s `max_depth` limit.

---

## Class summary

- A perceptron is a weighted sum plus an activation function — with a sigmoid activation, mathematically the same model as `NB07`'s Logistic Regression.
- Stacking layers only adds power if non-linear activation functions (ReLU, sigmoid, tanh) sit between them.
- Training = forward pass (predict) → loss (how wrong?) → backpropagation (whose fault?) → gradient descent (nudge weights to improve).
- We built, trained, and evaluated a real PyTorch MLP on the same Sonar dataset and train/test split as `NB08`, making the comparison to classical ML direct and fair.
- A first, untuned neural network doesn't automatically beat well-tuned classical models — depth and flexibility have to be earned through proper training practice, which is next.

## For the next class (NB12)

We'll train deep networks properly: validation curves during training, regularization techniques (dropout, early stopping) to fight overfitting, and a look at different optimizers — turning today's "it runs" network into one you'd actually trust.

## Homework / Practice Ideas

1. Change the hidden layer sizes (e.g., `128` and `64` instead of `32` and `16`) — does test accuracy improve, get worse, or barely change?
2. Change `n_epochs` to 500 — does the training loss keep falling? Does test accuracy improve along with it, or does it plateau (or get worse) while training loss keeps dropping?
3. Replace `nn.ReLU()` with `nn.Tanh()` in the hidden layers — does training behave differently (check the loss curve shape)?
4. Try a different learning rate (`lr=0.01` and `lr=0.0001`) — relate what you see back to Part 5's explanation of what the learning rate controls.
5. In your own words, explain why we used `BCEWithLogitsLoss` (binary cross-entropy) here instead of the MAE/RMSE metrics from `NB07` — what kind of problem is each one meant for?

> **Further reading**: [PyTorch: `nn` module documentation](https://pytorch.org/docs/stable/nn.html) · [PyTorch basics tutorial](https://pytorch.org/tutorials/beginner/basics/intro.html).

> ***As always: a model that "runs without errors" and a model that "actually works well" are very different bars — today only gets us to the first one.***
